# 1 · Provision the index (required, run this first)

This is not a demo step — it's the thing that makes `mcp_server.py` and `web_app.py`
actually work. Both of them call `portfolio.load_all_indexes()` at startup, which
**requires** the index to already exist in Qdrant; neither one builds it themselves
anymore. This notebook is what builds it.

Fetches each project's real README (live from GitHub, falls back to a saved copy in
`data/` if GitHub isn't reachable), splits it into chunks, embeds them, and writes the
result to Qdrant — where it stays, persistently, for both services to load.

**Run this whenever:** you're setting this up for the first time, you add a new project
to `config.PROJECTS`, or you've edited a project's README and want the index to reflect
the change (pass `force=True` for that last case — see below).

In [ ]:
%%capture
!pip install -q -r requirements.txt


In [ ]:
# Get the project files (config.py, portfolio.py, data/) if they aren't
# already here -- lets this notebook be opened and run on its own in Colab.
import os, subprocess, sys

if not os.path.exists("portfolio.py"):
    if os.path.exists("../portfolio.py"):
        os.chdir("..")
    else:
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/hossamhamdy333/AI_Portfolio.git", "repo"],
            check=True,
        )
        os.chdir("repo/Codebase_Insight_Agent")

sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())


In [ ]:
from getpass import getpass

if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Google API key (Gemini): ")

# QDRANT_URL/QDRANT_API_KEY make the index PERSISTENT (Qdrant Cloud, free
# tier is enough) instead of rebuilt from scratch every time. This matters
# specifically because this notebook runs in a fresh Colab VM each time,
# separate from wherever mcp_server.py/web_app.py actually run - without a
# real, shared QDRANT_URL, this notebook's work never reaches those
# services at all. Get a free instance at https://cloud.qdrant.io
if not os.environ.get("QDRANT_URL"):
    os.environ["QDRANT_URL"] = input("Qdrant Cloud URL (blank = local in-memory, no persistence): ")
if os.environ["QDRANT_URL"] and not os.environ.get("QDRANT_API_KEY"):
    os.environ["QDRANT_API_KEY"] = getpass("Qdrant API key: ")


Build the indexes. `force=False` (the default) skips any project that's already
indexed — safe to re-run after adding one new project without re-embedding the other 10.
Use `force=True` after editing an existing README, so the stale old chunks are actually
replaced, not left sitting there alongside the new ones.

In [ ]:
import config
import portfolio

indexes = portfolio.build_all_indexes(force=False)
print(f"\n{len(indexes)} / {len(config.PROJECTS)} projects indexed")

if not os.environ.get("QDRANT_URL"):
    print("\nWARNING: QDRANT_URL was left blank, so this used a local in-memory Qdrant. "
          "Nothing was actually persisted - mcp_server.py and web_app.py running "
          "elsewhere will NOT see this index. Re-run with a real Qdrant Cloud URL for "
          "this to actually matter.")


Quick sanity check — ask one project's index a direct question:

In [ ]:
engine = indexes["Credit_Fraud_Detection"].as_query_engine(similarity_top_k=3)
print(engine.query("What model performed best and why?"))
